Big Picture: Data is a table of molecules, each row is a molecule, bunch of numerical desciptors (features), the number attempting to predict is the dft_barrier - reaction barrier that was predicted. Aim is to learn a cheap statistical model that predicts the DFT number from the features.

Regression problem (continuous target, not a class), model chosen is Support vector regression (SVR) with an RBF kernal.

Script has four jobs, everything in it serves one of the following 

1 - Get the data and split it into features X and target Y, honouring the pre-assigned train/test split. 
2 - Define the model - the pipeline: clean -> filter -> scale -> SVR
3 - Estimate how good a given set of hyperparameters is using cross-validation on the training data only
4 - Lock the test set in a vault until the very end, and only unlock it once 

Feature X and target Y: X is the input to the function, Y is the result. X is 2D, a matrix of bunch of features (columns) and the rows of tons of molecules. The taregt y (dft value) is simply a 1D column. 
X is what the model sees and Y is what the model is graded against. During the training it looks at X, makes and guess and then checks it against Y, and adjusts. The learning is that "guess,check and adjust" loop. 

In [14]:
#Import necessary libraries

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [15]:
#Loading the data and honouring the split 

data = pd.read_csv("data/generated_features.csv")
train = data[data["split"] == "train"].copy()
test  = data[data["split"] == "test"].copy()

Have the train and the test Dataframes, now need to get the X_train and the y_train out of the train first. So need to remove the feature columns from the data leaving just the data that is required.

- dft_barrier removed from the data as it shows the answer (y). If sat in X the model woudl know the answer already. 
- smiles is removed because it is a text label and not a number, SVR can't work on a string 
- file_idx and split - same deal as above 

X_train grabs the descriptor columns -> the grid of facts
y_train grabs just the dft_barrier column -> the list of answers

In [16]:
non_features = {"file_idx","smiles","split","dft_barrier"}

#list comprehension, building a new list by walking through an existing collection and keeping/transforming items. Loop that produces a list.

feature_cols = [c for c in data.columns if c not in non_features]

X_train = train[feature_cols]
y_train = train["dft_barrier"]


Now buidling the pipeline - the list of steps that data flows through in order, where each step transforms the data and hands it over to the next. 
- Checking if there is any gaps in the data that could disrupt the pipeline 


In [17]:
X_train.isna().sum().sum()
#This means there is 2658 missing cells across the data hence need to sort this out before training the model.

np.int64(2658)

In [18]:
#need to add an imputer to the pipeline to handle missing values. 
#The question is what type of inputer to use. The most common are mean, median, and most frequent.
#SimpleImputer is a class in sklearn that provides basic strategies for imputing missing values.
#need to add the strategy parameter to the SimpleImputer class to specify how to impute the missing values.

imputer = SimpleImputer(strategy='median')

#Then hand the data to the tool after creating the tool 

imputer.fit_transform(X_train)


#fit learns something and transform applies the learned information to the data.


array([[27.61779935,  1.        ,  0.        , ..., -0.425584  ,
         0.47825033,  0.508782  ],
       [31.39767769,  1.        ,  1.        , ..., -0.475475  ,
         0.484222  ,  0.497019  ],
       [32.63512658,  1.        ,  1.        , ..., -0.46397   ,
         0.47506233,  0.489191  ],
       ...,
       [14.70020102,  1.        ,  1.        , ..., -0.441176  ,
         0.47829033,  0.505877  ],
       [16.24521738,  1.        ,  0.        , ..., -0.437357  ,
         0.49157867,  0.522344  ],
       [15.50212761,  1.        ,  0.        , ..., -0.403102  ,
         0.47696767,  0.519472  ]], shape=(891, 144))

In [19]:
#A model can only learn from a change in the data, so if a column has zero varience then the model will not learn anything from that column 
#Hence that column needs to be removed from the data

varience_threshold = VarianceThreshold()

#The default threshold is 0, which means that all features with zero variance will be removed.
#Then need to apply this tool to the data

varience_threshold.fit_transform(X_train)



array([[27.61779935,  1.        ,  0.        , ..., -0.425584  ,
         0.47825033,  0.508782  ],
       [31.39767769,  1.        ,  1.        , ..., -0.475475  ,
         0.484222  ,  0.497019  ],
       [32.63512658,  1.        ,  1.        , ..., -0.46397   ,
         0.47506233,  0.489191  ],
       ...,
       [14.70020102,  1.        ,  1.        , ..., -0.441176  ,
         0.47829033,  0.505877  ],
       [16.24521738,  1.        ,  0.        , ..., -0.437357  ,
         0.49157867,  0.522344  ],
       [15.50212761,  1.        ,  0.        , ..., -0.403102  ,
         0.47696767,  0.519472  ]], shape=(891, 143))

In [20]:
#Now for the scaling of the data, each feature has a different scale so they need to be scaled to be comparable, standard coding in maths 
#The formula for standard scaling is (x - mean) / std, where x is the feature value, mean is the mean of the feature, and std is the standard deviation of the feature.
#Non negotiable for SVR as it calculates distances between points in the feature space, and if the features are not scaled, the distances will be dominated by the features with larger scales.

scaler = StandardScaler()

scaler.fit_transform(X_train)


array([[ 0.71823296, -0.37139068, -0.77807802, ..., -0.78172839,
         0.97729373, -0.16748992],
       [ 1.31669463, -0.37139068,  1.28521816, ..., -1.63737592,
         1.34633801, -0.59918825],
       [ 1.51261779, -0.37139068,  1.28521816, ..., -1.44006128,
         0.78027785, -0.88647334],
       ...,
       [-1.32698826, -0.37139068,  1.28521816, ..., -1.04913647,
         0.9797657 , -0.27410249],
       [-1.08236846, -0.37139068, -0.77807802, ..., -0.98363932,
         1.80097419,  0.33023113],
       [-1.20002059, -0.37139068, -0.77807802, ..., -0.39615449,
         0.89802594,  0.22482965]], shape=(891, 144))

In [21]:
svr = SVR(kernel = 'rbf',C=5,epsilon=0.1,gamma=0.01)

#kernel controls the strategy by which the model will measure the similarity between the molecules, while the other parameters control how aggressivley the model behaves 
#The kernel is the function that takes two molecules and then compares how simlar they are and returns a number. The aim of the svr/ the job is to answer "how simliar are these two molecules"
#The model predicts the barrier by asking how simliar it is to the next molecule so it is a comparision model 
#If two molecules are identical there distance between eachother in the feature space is zero and hence the rbf returns 1, maximum similarity 

svr

,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.01
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",5
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [25]:
#Now to build the pipeline that brings all of these together, Pipeline was imported it takes one main argument "steps" an ordered list of the stations
#Order is important as the data flows through the order, the format that the pipeline adpots also is uniqye it has ("name",tool) where the name is any short string label for the tool
#tool is one of the four objects that was made above.
#Is a list so remeber to add commas between the steps and the last step doesn not need a comma after it.

estimator = Pipeline (steps = [
    ("imputer",imputer),
    ("varience",varience_threshold),
    ("scaler",scaler),
    ("svr",svr)
])

variable

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('varience', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto acc

In [23]:
#Now for the cross validation, so now is the part where we see how good the model actually is 
#Cannot answer this by training the same configurations (parameters set to what they are now) on all test data, need to find a way to estimate the performance on only the test data
#This can be done with k-fold cross-validation, using the two tools imported, kFold and corss_validate 
#Given the current paramaters, how do we choose what to change them to next?? Need a search strategy 

kfold = KFold(n_splits = 5, shuffle = True, random_state = 0)
kfold

KFold(n_splits=5, random_state=0, shuffle=True)

In [ ]:
#Now for the 'engine' the cross_validate, this is the piece that actually does the five rounds, everything so far has been construction
#Estimator is the thing that is trying to be fitted and scored, the assembled pipeline.
#Shows the results (5) from the 5 fold cross validation
results = cross_validate(
    estimator = estimator, 
    X = X_train,
    y = y_train,
    cv = kfold,
    scoring = {"mae": "neg_mean_absolute_error",
               "rmse":"neg_root_mean_squared_error",
               "r_squared": "r2"}


)


results

{'fit_time': array([0.03079772, 0.02865434, 0.02671099, 0.02694845, 0.02755117]),
 'score_time': array([0.01145959, 0.01063466, 0.00989389, 0.01025748, 0.01005483]),
 'test_mae': array([-1.35407941, -1.3924172 , -1.24747118, -1.3176412 , -1.29504594]),
 'test_rmse': array([-1.82910904, -1.89527567, -1.70662712, -1.73330437, -1.84565665]),
 'test_r_squared': array([0.87634346, 0.89082616, 0.89813061, 0.90162683, 0.88415969])}

In [ ]:
#The inner loop is now complete, raw csv -> features/target split -> four-station pipeline -> five-fold CV -> six summary numbers

fold_mae = -np.mean(results["test_mae"])
fold_std_mae = np.std(results["test_mae"],ddof=1)

fold_rmse = -np.mean(results["test_rmse"])
fold_std_rmse = np.std(results["test_rmse"],ddof=1)

fold_r2 = np.mean(results["test_r_squared"])
fold_std_r2 = np.std(results["test_r_squared"],ddof=1)


print(f"MAE = {np.mean(fold_mae):.4f}")
print(f"STD_MAE = {np.mean(fold_std_mae):.4f}")

print(f"RMSE = {np.mean(fold_rmse):.4f}")
print(f"STD_RMSE = {np.mean(fold_std_rmse):.4f}")

print(f"r2 = {np.mean(fold_r2):.4f}")
print(f"STD_r2 = {np.mean(fold_std_r2):.4f}")

#all done 


MAE = 1.3213
STD_MAE = 0.0554
RMSE = 1.8020
STD_RMSE = 0.0793
r2 = 0.8902
STD_r2 = 0.0103
